In [8]:
%history -g -f recovered_code.py

In [9]:
import sqlite3
import os

# Find the IPython history database path
# Usually ~/.ipython/profile_default/history.sqlite on Mac/Linux
# Or C:\Users\YourName\.ipython\profile_default\history.sqlite on Windows
db_path = os.path.expanduser('~/.ipython/profile_default/history.sqlite')

if os.path.exists(db_path):
    con = sqlite3.connect(db_path)
    cursor = con.cursor()
    
    # Grab the last 100 executed commands
    cursor.execute("SELECT source FROM history ORDER BY session DESC, line DESC LIMIT 100")
    
    rows = cursor.fetchall()
    
    # Print them out (reversing them so they read in chronological order)
    for row in reversed(rows):
        print(row[0])
        print("#" + "-"*40)
else:
    print(f"Could not find the database at {db_path}")

import pandas as pd
import numpy as np
import scipy.stats as stats

def run_rolling_accumulation_test(csv_path, sigma_threshold=1.0):
    """
    Tests rolling 3-day accumulation regimes across overlapping periods.
    Evaluates mean reversion over the immediate subsequent 2-day and 5-day windows.
    """
    print(f"Loading daily data from: {csv_path}")
    raw_df = pd.read_csv(csv_path, sep=None, engine='python')
    
    # 1. Structural Mapping & Chronological Sorting (Using correct 1 and 4 indices)
    df = pd.DataFrame()
    df['Date'] = pd.to_datetime(raw_df.iloc[:, 0].str.replace('.', '-', regex=False))
    df['Open'] = raw_df.iloc[:, 1].astype(float)
    df['Close'] = raw_df.iloc[:, 4].astype(float)
    df = df.sort_values(by='Date').reset_index(drop=True)
    df = df.drop_duplicates(subset=['Date']).reset_index(drop=True)
    
    # 2. Calculate Log Returns for mathematically accurate multi-day compounding
    df['Daily_Log_Return'] = np.log(df['Close'] / df['Open'])
    
    